# D-MeZO — Bootstrap notebook for Google Colab Pro+

Target hardware: **RTX PRO 6000 Blackwell (96 GB)** или A100 80 GB.

Этот ноутбук:
1. Монтирует Google Drive (для чекпоинтов).
2. Клонирует или копирует проект `dmezo`.
3. Устанавливает зависимости.
4. Проверяет GPU.
5. Запускает Day 1 sanity check (MeZO на Qwen3-4B / SST-2).

**Бюджет:** ~30-50 compute units на полный прогон Day 1.

## 0. Mount Google Drive для чекпоинтов

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUNS_DIR = '/content/drive/MyDrive/dmezo_runs'
os.makedirs(RUNS_DIR, exist_ok=True)
print(f'Runs will be saved to: {RUNS_DIR}')

## 1. Клонировать проект из GitHub

Установи `GH_REPO` ниже на свой приватный repo (e.g. `your-username/dmezo`). Если repo приватный, добавь Personal Access Token через Colab Secrets (key icon в sidebar): создай secret `GH_TOKEN` с `repo` scope, в нём — твой PAT.

In [ ]:
GH_REPO = 'Siesher/dmezo'  # private repo on github.com

import os, shutil
if os.path.exists('/content/dmezo'):
    shutil.rmtree('/content/dmezo')

# Try to read PAT from Colab Secrets (for private repos).
try:
    from google.colab import userdata
    gh_token = userdata.get('GH_TOKEN')
    clone_url = f'https://{gh_token}@github.com/{GH_REPO}.git'
except Exception:
    clone_url = f'https://github.com/{GH_REPO}.git'  # works only if repo is public

!git clone {clone_url} /content/dmezo
%cd /content/dmezo
!git log --oneline -5

## 2. Установить зависимости

In [ ]:
!pip install -q --upgrade pip
!pip install -q -e .
# Flash attention (опционально, ускоряет MeZO forward на Blackwell):
!pip install -q flash-attn --no-build-isolation || echo 'flash-attn install skipped'

## 3. Проверить GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    total_gb = torch.cuda.mem_get_info()[1] / 1e9
    print(f'GPU: {name}')
    print(f'Memory: {free_gb:.1f} / {total_gb:.1f} GB free')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')
    cc = torch.cuda.get_device_capability(0)
    print(f'Compute capability: {cc} (Blackwell = (10, 0) or higher)')

## 4. Запустить тесты

Должны пройти все: perturbation determinism + topology properties.

In [ ]:
!pytest tests/ -v

## 5. Day 1 sanity check — MeZO на Qwen3-4B / SST-2

Скрипт пишет JSONL-лог и чекпоинты в `experiments/01_sanity_qwen3_4b_sst2/` (внутри `/content/dmezo`). Чтобы сохранить в Drive, передадим `--output-dir`.

In [ ]:
OUTPUT_DIR = f'{RUNS_DIR}/01_sanity_qwen3_4b_sst2'
MLFLOW_URI = f'file:{RUNS_DIR}/mlruns'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_4b_sst2.yaml \
    --output-dir {OUTPUT_DIR} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-4b-sst2-day1

## 6. Анализ результата

## 6a. MLflow run summary

MLflow data persists in `{RUNS_DIR}/mlruns/` (на Drive). Можно посмотреть локально через `mlflow ui --backend-store-uri file:./mlruns` после скачивания, или прямо здесь в ноутбуке:

In [ ]:
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_sanity')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC'])
display(runs[['run_id', 'tags.mlflow.runName', 'tags.sanity_verdict',
              'metrics.init_eval_loss', 'metrics.final_eval_loss', 'metrics.eval_loss_drop_pct',
              'params.model.name', 'params.mezo.lr', 'params.mezo.eps']].head(10))

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

log_path = f'{OUTPUT_DIR}/log.jsonl'
rows = [json.loads(line) for line in open(log_path)]
df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train = df[df['train_loss'].notna()] if 'train_loss' in df.columns else df.iloc[0:0]
ev = df[df.get('eval_loss').notna()] if 'eval_loss' in df.columns else df.iloc[0:0]
if not train.empty:
    axes[0].plot(train['step'], train['train_loss'], label='train (moving avg)')
if not ev.empty:
    axes[0].plot(ev['step'], ev['eval_loss'], 'o-', label='eval')
axes[0].set_xlabel('MeZO step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Day 1 sanity: MeZO on Qwen3-4B / SST-2')
axes[0].legend()
axes[0].grid(alpha=0.3)

if 'projected_grad' in df.columns:
    pg = df[df['projected_grad'].notna()]
    axes[1].plot(pg['step'], pg['projected_grad'])
    axes[1].set_xlabel('MeZO step')
    axes[1].set_ylabel('Projected gradient')
    axes[1].set_title('Projected gradient (should be non-vanishing, non-NaN)')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sanity_plot.png', dpi=120)
plt.show()
print('Saved plot to', f'{OUTPUT_DIR}/sanity_plot.png')

## 7. Что дальше

Если sanity прошёл (eval loss упал на >10%):
- Переходить к Day 2 чтению FedKSeed/FedZeN.
- Запустить centralized baselines на BoolQ/COPA фоном.

Если sanity провалился:
- Проверить hyperparameters: `lr` в диапазоне 1e-7 - 1e-5, `eps=1e-3`.
- Проверить, что `param.requires_grad=True` для всех параметров.
- Распечатать `model.config` — убедиться, что Qwen3-4B загрузился именно как ожидалось.
- Запустить debug-версию с 10 steps и руками посмотреть на projected_grad.

## 8. Bonus: Qwen3.5-4B-Base — hybrid linear-attention sanity

Qwen3.5-4B-Base — это **vision-language модель с hybrid linear-attention text decoder** (32 слоя: 24 linear + 8 full attention, 3:1 ratio). Loader заморозит vision tower; MeZO будет perturbать только text decoder (~4.4B trainable params).

**Зачем:** Princeton MeZO 2023 проверил только на full-attention моделях. Это первый известный нам test MeZO на hybrid linear-attention арх. Любой результат публикабелен.

Если уже клонировал proj и хочешь подтянуть свежий код — сделай `!cd /content/dmezo && git pull` перед запуском ячейки ниже.

In [ ]:
OUTPUT_DIR_Q35 = f'{RUNS_DIR}/01_sanity_qwen3_5_4b_base_sst2'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_5_4b_base_sst2.yaml \
    --output-dir {OUTPUT_DIR_Q35} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-5-4b-base-sst2-hybrid-arch

In [ ]:
# Side-by-side: Qwen3-4B (Day 1) vs Qwen3.5-4B-Base (hybrid)
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_sanity')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC'])
cols = ['tags.mlflow.runName', 'params.model.name', 'tags.sanity_verdict',
        'metrics.init_eval_loss', 'metrics.final_eval_loss', 'metrics.eval_loss_drop_pct']
display(runs[cols].head(10))

## 9. Cross-dataset check — BoolQ on both architectures

После SST-2 Day 1 (Qwen3-4B PASS 88%) и Section 8 (Qwen3.5-4B-Base hybrid PASS 94.7%) — закрепляем finding на **BoolQ** (yes/no QA на длинных passages, SuperGLUE). Если оба тоже PASS — у нас 2×2 design (architecture × task), который перекрывает основной paper-claim risk "это специфика SST-2".

Сначала `!cd /content/dmezo && git pull` если notebook уже клонирован, иначе перезапусти runtime и re-run cells 1-6.

In [ ]:
!cd /content/dmezo && git pull

# Qwen3-4B / BoolQ (full attention baseline)
OUTPUT_DIR_Q3_BQ = f'{RUNS_DIR}/02_sanity_qwen3_4b_boolq'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_4b_boolq.yaml \
    --output-dir {OUTPUT_DIR_Q3_BQ} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-4b-boolq

In [ ]:
# Qwen3.5-4B-Base / BoolQ (hybrid linear-attention)
OUTPUT_DIR_Q35_BQ = f'{RUNS_DIR}/02_sanity_qwen3_5_4b_base_boolq'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_5_4b_base_boolq.yaml \
    --output-dir {OUTPUT_DIR_Q35_BQ} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-5-4b-base-boolq-hybrid

## 10. Day 4 — first federated D-MeZO (2 clients, complete graph, weight_avg)

После centralized PASS на Qwen3-4B (Day 1, drop 88.1%) и кросс-арх validation (Section 8-9) — главный путь paper'а: **decentralized federated MeZO**. 2 клиента, complete topology, IID SST-2, weight-averaging consensus.

**Success criterion** (Week 1 plan): federated final eval ≤ centralized Day 1 × 1.1 = 0.18.

Memory budget: 2 × Qwen3-4B bf16 = 16 GB weights + activations ~25 GB peak. Blackwell 96 GB handles. Сделай `git pull` перед ячейкой.

In [ ]:
!cd /content/dmezo && git pull

OUTPUT_DIR_DAY4 = f'{RUNS_DIR}/03_dmezo_2c_complete_qwen3_4b_sst2'
!python scripts/03_dmezo_federated.py \
    --config configs/dmezo_2c_complete_qwen3_4b_sst2.yaml \
    --output-dir {OUTPUT_DIR_DAY4} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-4b-sst2-2c-complete-weight-avg

In [ ]:
# Day 4 vs Day 1 comparison: federated vs centralized on same model+task
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
runs_all = []
for exp_name in ['dmezo_sanity', 'dmezo_federated']:
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp is None:
        continue
    runs_all.append(mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC']))
import pandas as pd
runs = pd.concat(runs_all, ignore_index=True)
cols = ['tags.mlflow.runName', 'params.model.name', 'params.data.task',
        'tags.algo', 'tags.n_clients', 'tags.topology', 'tags.consensus_mode',
        'tags.sanity_verdict', 'metrics.init_eval_loss', 'metrics.final_eval_loss',
        'metrics.eval_loss_drop_pct']
display(runs[[c for c in cols if c in runs.columns]].head(15))

In [ ]:
# 2x2 design summary: architecture x task
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_sanity')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC'])
cols = ['tags.mlflow.runName', 'params.model.name', 'params.data.task',
        'tags.sanity_verdict', 'metrics.init_eval_loss', 'metrics.final_eval_loss',
        'metrics.eval_loss_drop_pct']
display(runs[cols].head(10))

## 11. Day 5 — federated D-MeZO grid (4 clients × topology × partition)

Цель Day 5 — впервые показать, что **federated MeZO работает на hybrid
linear-attention LLM** (Qwen3.5-4B-Base) при 4 клиентах с разными топологиями
и non-IID распределениями данных.

**Дизайн 2×2 grid** (4 запуска):

| topology | partition | rationale |
|---|---|---|
| complete(4) | IID | baseline — почти centralized (rho=0) |
| complete(4) | Dirichlet(0.5) | partition stress — Hsu 2019 convention |
| ring(4) | IID | topology stress — Koloskova 2020 rho~0.5 |
| ring(4) | Dirichlet(0.5) | worst cell — slow mixing + label skew |

Все runs: 1000 раундов, lr=3e-7, weight_avg consensus.

**Compute estimate:** 4 × ~10-12 min = ~45-50 min total, ~7-10 units.


In [ ]:
!cd /content/dmezo && git pull

CONFIGS = [
    'configs/dmezo_4c_complete_iid_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_complete_dir05_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_ring_iid_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2.yaml',
]
MLFLOW_URI = f'file:{RUNS_DIR}/mlruns'

for cfg_path in CONFIGS:
    # OUTPUT_DIR derives from the config's `output_dir` field, but we override
    # to land each run under Drive so it survives session loss.
    short = cfg_path.split('/')[-1].replace('.yaml', '')
    output_dir = f'{RUNS_DIR}/{short}'
    print(f'
=== {short} ===')
    !python scripts/03_dmezo_federated.py         --config {cfg_path}         --output-dir {output_dir}         --mlflow-uri {MLFLOW_URI}         --run-name {short}


In [ ]:
# Day 5 summary table — load all 4 runs from MLflow and compare.
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_federated_day5')
if exp is None:
    print('No experiment found — did the runs complete?')
else:
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    cols = [
        'tags.mlflow.runName',
        'params.federated.topology',
        'params.data.partition_mode',
        'metrics.init_eval_loss',
        'metrics.final_eval_loss',
        'metrics.eval_loss_drop_pct',
        'tags.sanity_verdict',
    ]
    cols_present = [c for c in cols if c in runs.columns]
    summary = runs[cols_present].sort_values('tags.mlflow.runName').reset_index(drop=True)
    # Compare against Day 4 centralized-equivalent (final 0.1793 on Qwen3-4B 2c).
    # Success criterion for Day 5: hybrid 4c <= 1.5 * Day 4 baseline = 0.27.
    DAY4_BASELINE = 0.1793
    DAY5_THRESHOLD = DAY4_BASELINE * 1.5
    print(f'Day 4 baseline (Qwen3-4B 2c complete IID): {DAY4_BASELINE:.4f}')
    print(f'Day 5 pass threshold (1.5x baseline):      {DAY5_THRESHOLD:.4f}')
    print()
    print(summary.to_string(index=False))


## 12. Day 6 — Nesterov-MeZO ablation (worst Day 5 cell)

Тестируем "N" в D-MeZO-N. Запускаем 2 варианта Nesterov-momentum на самом
сложном cell Day 5 (ring(4) + Dirichlet(α=0.5)):

- **Heavy-ball β=0.9** — стандартный momentum coefficient
- **Heavy-ball β=0.5** — пол-step momentum (safer для noisy ZO-grads)

Контроль = **Day 5 cell 4** (seed=42, без Nesterov, final=0.1381). Тот же seed
во всех Day 6 runs гарантирует bit-exact identical partition / MeZO seeds /
init — отличается ТОЛЬКО Nesterov ветвь.

**Гипотеза** (из `docs/03-algorithm-spec.md`): на heterogeneous partition со
slow consensus mixing Nesterov уменьшает client drift variance и даёт ~5-15%
улучшение. Falsifiable: если final ≥ 0.1381 — H0 не отвергаем.

Compute: 2 × ~37 мин ≈ 1.25 ч, ~5-7 units.


In [ ]:
!cd /content/dmezo && git pull

NEST_CONFIGS = [
    'configs/dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_nest09.yaml',
    'configs/dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_nest05.yaml',
]
MLFLOW_URI = f'file:{RUNS_DIR}/mlruns'

for cfg_path in NEST_CONFIGS:
    short = cfg_path.split('/')[-1].replace('.yaml', '')
    output_dir = f'{RUNS_DIR}/{short}'
    print(f'
=== {short} ===')
    !python scripts/03_dmezo_federated.py         --config {cfg_path}         --output-dir {output_dir}         --mlflow-uri {MLFLOW_URI}         --run-name {short}


In [ ]:
# Day 6 Nesterov ablation — compare against Day 5 cell 4 control.
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)
CONTROL_FINAL = 0.1381   # Day 5 cell 4 (ring + Dir(0.5), nesterov off, seed=42)

day6 = mlflow.get_experiment_by_name('dmezo_federated_day6')
if day6 is None:
    print('No Day 6 experiment found yet.')
else:
    runs = mlflow.search_runs(experiment_ids=[day6.experiment_id])
    cols = [
        'tags.mlflow.runName',
        'params.nesterov.beta',
        'metrics.final_eval_loss',
        'metrics.eval_loss_drop_pct',
    ]
    cols_present = [c for c in cols if c in runs.columns]
    df = runs[cols_present].sort_values('tags.mlflow.runName').reset_index(drop=True)

    print(f'Control (Day 5 cell 4, nesterov OFF): {CONTROL_FINAL:.4f}')
    print()
    print(df.to_string(index=False))
    print()
    # Compute relative improvement per row.
    if 'metrics.final_eval_loss' in df.columns:
        for _, row in df.iterrows():
            final = row['metrics.final_eval_loss']
            improvement_pct = (CONTROL_FINAL - final) / CONTROL_FINAL * 100
            verdict = 'HELPS' if improvement_pct > 2 else ('NEUTRAL' if improvement_pct > -2 else 'HURTS')
            beta = row.get('params.nesterov.beta', '?')
            print(f"  beta={beta}: final={final:.4f}  delta={improvement_pct:+.2f}%  -> {verdict}")
